# Advanced Statistical Analysis & Hypothesis Testing
## Synthetic Demonstration — Student Exam Scores

**Purpose:** Demonstrate normality tests, parametric and non-parametric two-sample tests, one-way ANOVA, two-way ANOVA, Tukey HSD post-hoc comparisons, confidence intervals, and distribution plots.

**Important:** This notebook uses a reproducible **synthetic dataset** created in the notebook. The results are for demonstrating the workflow only; they are not real scientific/business findings and should not be presented as real-world evidence. Replace the synthetic dataset with a documented real dataset before making substantive claims.

### Research question
Do average exam scores differ by teaching method, and are scores associated with both teaching method and study-time group?

### Hypotheses
- **Two-sample tests:** H0: the two population distributions/means do not differ (for t-test, equal means); H1: they differ.
- **One-way ANOVA:** H0: all teaching-method population means are equal; H1: at least one mean differs.
- **Two-way ANOVA:** H0 for teaching method: no teaching-method main effect; H0 for study-time group: no study-time main effect; H0 for interaction: no interaction between teaching method and study-time group.
- **Normality:** H0: observations follow a normal distribution; H1: observations do not follow a normal distribution.

### Analysis conventions
- Significance threshold: alpha = 0.05.
- All hypothesis tests are two-sided where applicable.
- Report effect estimates and 95% confidence intervals, not only p-values.
- Shapiro–Wilk and one-sample Kolmogorov–Smirnov against a normal distribution fitted using sample mean/SD are included as diagnostics. The usual one-sample KS p-value is not exact when parameters are estimated from the same sample; use it as an illustrative diagnostic, not a definitive calibrated test. A Lilliefors-corrected test would be preferable for formal inference.
- Synthetic fixture seed is fixed for reproducibility.


In [ ]:
# If needed, install dependencies in your environment:
# %pip install numpy pandas scipy statsmodels matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display

SEED = 20261021
rng = np.random.default_rng(SEED)
ALPHA = 0.05
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 30)
print("Seed:", SEED)
print("Alpha:", ALPHA)


## 1. Create a synthetic dataset
This simulated data is intentionally used to test the analysis workflow. It is not collected from real students.

In [ ]:
# Balanced synthetic data: 3 teaching methods x 2 study-time groups x 20 students per cell
methods = ["Traditional", "Blended", "Active"]
study_groups = ["Lower study time", "Higher study time"]
method_effect = {"Traditional": 0.0, "Blended": 4.0, "Active": 7.0}
study_effect = {"Lower study time": 0.0, "Higher study time": 5.0}

rows = []
for method in methods:
    for study_group in study_groups:
        mean = 65 + method_effect[method] + study_effect[study_group]
        scores = rng.normal(loc=mean, scale=8.0, size=20)
        for score in scores:
            rows.append({
                "teaching_method": method,
                "study_group": study_group,
                "exam_score": float(np.clip(score, 0, 100))
            })

df = pd.DataFrame(rows)
print("Shape:", df.shape)
display(df.head())
display(df.groupby(["teaching_method", "study_group"])["exam_score"].agg(["count", "mean", "std"]).round(2))


## 2. Descriptive statistics and distribution plots

In [ ]:
display(df.groupby("teaching_method")["exam_score"].describe().round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method in methods:
    axes[0].hist(df.loc[df.teaching_method == method, "exam_score"], bins=12, alpha=0.45, label=method)
axes[0].set_title("Exam-score distributions by teaching method")
axes[0].set_xlabel("Exam score")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[1].boxplot([df.loc[df.teaching_method == m, "exam_score"] for m in methods], labels=methods)
axes[1].set_title("Exam scores by teaching method")
axes[1].set_ylabel("Exam score")
plt.tight_layout()
plt.show()


## 3. Normality diagnostics: Shapiro–Wilk and Kolmogorov–Smirnov

In [ ]:
def normality_tests(values, label):
    values = pd.Series(values).dropna().astype(float)
    shapiro = stats.shapiro(values)
    mu, sigma = values.mean(), values.std(ddof=1)
    if sigma == 0:
        ks_stat, ks_p = np.nan, np.nan
    else:
        # Diagnostic only: standard KS p-value is not calibrated for estimated mu/sigma.
        ks = stats.kstest(values, "norm", args=(mu, sigma))
        ks_stat, ks_p = ks.statistic, ks.pvalue
    return {
        "group": label, "n": len(values), "mean": mu, "sd": sigma,
        "Shapiro_W": shapiro.statistic, "Shapiro_p": shapiro.pvalue,
        "KS_D": ks_stat, "KS_p_diagnostic_only": ks_p
    }

normality = pd.DataFrame([
    normality_tests(df.loc[df.teaching_method == method, "exam_score"], method)
    for method in methods
])
display(normality.round(4))
print("Interpretation: p < 0.05 suggests evidence against normality; p >= 0.05 does not prove normality.")


## 4. Two-sample t-test and Mann–Whitney U test
Compare Traditional vs Active teaching methods. Welch's t-test does not assume equal variances.

In [ ]:
group_a = df.loc[df.teaching_method == "Traditional", "exam_score"].to_numpy()
group_b = df.loc[df.teaching_method == "Active", "exam_score"].to_numpy()

# Welch two-sample t-test
tt = stats.ttest_ind(group_b, group_a, equal_var=False)
n_a, n_b = len(group_a), len(group_b)
mean_diff = group_b.mean() - group_a.mean()
se = np.sqrt(group_a.var(ddof=1)/n_a + group_b.var(ddof=1)/n_b)
df_welch = (group_a.var(ddof=1)/n_a + group_b.var(ddof=1)/n_b)**2 / (
    (group_a.var(ddof=1)/n_a)**2/(n_a-1) + (group_b.var(ddof=1)/n_b)**2/(n_b-1)
)
ci_t = stats.t.interval(0.95, df_welch, loc=mean_diff, scale=se)
mw = stats.mannwhitneyu(group_b, group_a, alternative="two-sided")
print(f"Traditional mean = {group_a.mean():.2f} (n={n_a})")
print(f"Active mean = {group_b.mean():.2f} (n={n_b})")
print(f"Mean difference (Active - Traditional) = {mean_diff:.2f}")
print(f"95% CI for mean difference = [{ci_t[0]:.2f}, {ci_t[1]:.2f}]")
print(f"Welch t-test: t={tt.statistic:.3f}, p={tt.pvalue:.6g}")
print(f"Mann–Whitney U: U={mw.statistic:.3f}, p={mw.pvalue:.6g}")
print("Decision rule: reject the relevant null hypothesis when p < 0.05; interpret alongside effect size and assumptions.")


## 5. One-way ANOVA and Tukey HSD post-hoc analysis

In [ ]:
oneway_model = ols("exam_score ~ C(teaching_method)", data=df).fit()
oneway_table = sm.stats.anova_lm(oneway_model, typ=2)
display(oneway_table)
tukey = pairwise_tukeyhsd(endog=df["exam_score"], groups=df["teaching_method"], alpha=ALPHA)
print(tukey)


## 6. Two-way ANOVA (teaching method × study-time group)

In [ ]:
twoway_model = ols("exam_score ~ C(teaching_method) * C(study_group)", data=df).fit()
twoway_table = sm.stats.anova_lm(twoway_model, typ=2)
display(twoway_table)
print("Main effects and interaction are tested separately. Interpret the interaction before simplifying main effects.")


## 7. Effect size and confidence interval notes

In [ ]:
# One-way ANOVA eta-squared: between-method sum of squares / total sum of squares
ss_between = oneway_table.loc["C(teaching_method)", "sum_sq"]
ss_total = oneway_table["sum_sq"].sum()
eta_squared = ss_between / ss_total
print(f"One-way ANOVA eta-squared = {eta_squared:.3f}")

# Cohen's d (pooled-SD descriptive effect size for Active vs Traditional)
pooled_sd = np.sqrt(((n_a-1)*group_a.var(ddof=1) + (n_b-1)*group_b.var(ddof=1)) / (n_a+n_b-2))
cohens_d = mean_diff / pooled_sd
print(f"Cohen's d (Active - Traditional) = {cohens_d:.3f}")
print("The Welch mean-difference CI is reported above. Tukey HSD reports multiplicity-adjusted pairwise comparisons.")


## 8. Findings template and interpretation rules

Complete this section only after running the notebook. Because the dataset is synthetic, findings describe only the simulated example.

- **Normality:** Review Shapiro–Wilk results per group. A non-significant result does not prove normality. The KS result here is diagnostic only because distribution parameters were estimated from the same sample.
- **Two-sample tests:** Report the Active-minus-Traditional mean difference, 95% CI, Welch t-test p-value, Mann–Whitney U p-value, and effect size.
- **One-way ANOVA:** If p < 0.05, evidence suggests not all teaching-method means are equal. Use Tukey HSD to inspect pairwise differences with family-wise error adjustment.
- **Two-way ANOVA:** Report teaching-method, study-group, and interaction terms. A significant interaction means the teaching-method association differs by study group.
- **Caution:** Statistical significance is not causality. The synthetic example is not evidence about real students.

## 9. Reproducibility and limitations

- Random seed: `20261021`
- Dataset: generated in notebook; no real or personal data.
- Planned follow-up: replace synthetic data with a documented real dataset, record source/licence/access date, check measurement definitions, and rerun the same analysis plan.
- Assumptions: independent observations, appropriate outcome scale, and model residual assumptions should be checked for real data.
- KS caution: for a formal normality test with estimated mean and standard deviation, use a Lilliefors-corrected test rather than interpreting the standard KS p-value as exact.
